# Mini-projet: classification des fleurs Iris

## 1. Introduction

L'objectif de ce projet est de construire et de comparer plusieurs modèles de classification supervisée afin de prédire l'espèce d'une fleur Iris à partir de quatre caractéristiques : la longueur et la largeur des sépales, ainsi que la longueur et la largeur des pétales.

Le dataset utilisé provient de l'UCI Machine Learning Repository. Les trois espèces à distinguer sont *Iris setosa*, *Iris versicolor* et *Iris virginica*.

Le projet suivra un workflow complet de Machine Learning, de l'exploration des données jusqu'à l'évaluation comparative des modèles et à la conclusion critique.

In [ ]:
# Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement du dataset Iris depuis UCI
from ucimlrepo import fetch_ucirepo

iris = fetch_ucirepo(id=53)

# Récupération des caractéristiques et de la cible
X = iris.data.features.copy()
y = iris.data.targets.copy()

# Création du DataFrame complet
data = pd.concat([X, y], axis=1)

# Affichage des premières lignes
display(data.head())

## 2. Analyse exploratoire des données

Cette étape permet de comprendre la structure du dataset, de vérifier sa qualité et d'identifier les caractéristiques utiles à la classification des fleurs Iris. Des statistiques descriptives et des visualisations seront utilisées pour étudier les relations entre les variables et les espèces.

In [ ]:
# Dimensions du dataset
print("Dimensions du dataset :", data.shape)

# Informations sur les colonnes
data.info()

# Valeurs manquantes
print("\nValeurs manquantes :")
print(data.isnull().sum())

# Répartition des espèces
print("\nNombre de fleurs par espèce :")
print(data.iloc[:, -1].value_counts())

# Statistiques descriptives
print("\nStatistiques descriptives :")
display(data.describe())

In [ ]:
# Affichage des métadonnées du dataset
print(iris.metadata)

# Informations sur les variables
display(iris.variables)

## 3. Préparation des données

Les quatre mesures numériques constituent les variables explicatives, notées X. La colonne `species` représente la variable cible, notée y. L'objectif est d'entraîner les modèles à apprendre la relation entre les mesures d'une fleur et son espèce.

In [ ]:
# Renommage des colonnes pour faciliter leur utilisation
data.columns = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "species"
]

# Séparation des caractéristiques et de la cible
X = data[
    [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]
].copy()

y = data["species"].copy()

# Vérification
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nCaractéristiques :")
display(X.head())

print("\nCible :")
display(y.head())

In [ ]:
# Séparation des données en ensembles d'entraînement et de test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Taille des données d'entraînement :", X_train.shape)
print("Taille des données de test :", X_test.shape)

print("\nRépartition des espèces dans l'ensemble de test :")
print(y_test.value_counts())

**4. Visualisation des données**

Les visualisations permettent d'étudier la distribution des caractéristiques et les différences entre les trois espèces Iris.

In [ ]:
# Répartition des espèces
plt.figure(figsize=(7, 4))

sns.countplot(data=data, x="species")

plt.title("Répartition des fleurs par espèce")
plt.xlabel("Espèce")
plt.ylabel("Nombre de fleurs")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des caractéristiques selon l'espèce
colonnes_numeriques = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for colonne, axe in zip(colonnes_numeriques, axes.flatten()):
    sns.boxplot(data=data, x="species", y=colonne, ax=axe)
    axe.set_title(f"Distribution de {colonne}")
    axe.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Relations entre les caractéristiques
sns.pairplot(
    data,
    hue="species",
    diag_kind="hist"
)

plt.show()

Interprétation des visualisations

Les graphiques permettent de comparer les caractéristiques des trois espèces et d'observer les relations entre les variables numériques.

Les différences de longueur et de largeur des pétales peuvent notamment contribuer à distinguer les espèces.

## 5. Entraînement des modèles de classification

Trois algorithmes de classification supervisée seront comparés : la régression logistique, l'arbre de décision et Random Forest.

Les données d'entraînement seront utilisées pour apprendre les modèles. Les données de test seront conservées pour évaluer leurs performances sur des observations non utilisées pendant l'entraînement.


In [ ]:
# Importation des algorithmes et outils nécessaires
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Définition des modèles
modeles = {
    "Régression logistique": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ),

    "Arbre de décision": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

print("Nombre de modèles à comparer :", len(modeles))

In [ ]:
# Entraînement des modèles
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    print(f"{nom} : entraînement terminé")

## 6. Évaluation des modèles

Les modèles seront évalués à l'aide de l'accuracy, de la précision macro, du rappel macro et du F1-score macro.

Ces métriques permettront de comparer les performances globales des algorithmes et leur capacité à reconnaître chacune des trois espèces.

In [ ]:
# Importation des métriques
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Évaluation des modèles sur les données de test
resultats = []

for nom, modele in modeles.items():
    predictions = modele.predict(X_test)

    resultats.append({
        "Modèle": nom,
        "Accuracy": accuracy_score(y_test, predictions),
        "Précision macro": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Rappel macro": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "F1-score macro": f1_score(
            y_test, predictions, average="macro", zero_division=0
        )
    })

resultats_df = pd.DataFrame(resultats)

resultats_df = resultats_df.sort_values(
    by="F1-score macro",
    ascending=False
).reset_index(drop=True)

display(resultats_df.round(3))

In [ ]:
# Visualisation comparative des performances
resultats_long = resultats_df.melt(
    id_vars="Modèle",
    value_vars=[
        "Accuracy",
        "Précision macro",
        "Rappel macro",
        "F1-score macro"
    ],
    var_name="Métrique",
    value_name="Score"
)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=resultats_long,
    x="Modèle",
    y="Score",
    hue="Métrique"
)

plt.title("Comparaison des performances des modèles")
plt.xlabel("Modèle")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(title="Métrique", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Validation croisée

La validation croisée à trois plis permet d'évaluer la stabilité des modèles sur plusieurs découpages des données d'entraînement. Le F1-score macro moyen et son écart-type seront utilisés pour comparer les algorithmes.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Définition de la validation croisée
validation = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Évaluation par validation croisée
resultats_cv = []

for nom, modele in modeles.items():
    scores = cross_val_score(
        modele,
        X_train,
        y_train,
        cv=validation,
        scoring="f1_macro"
    )

    resultats_cv.append({
        "Modèle": nom,
        "F1 moyen": scores.mean(),
        "Écart-type": scores.std()
    })

resultats_cv_df = pd.DataFrame(resultats_cv)

resultats_cv_df = resultats_cv_df.sort_values(
    by="F1 moyen",
    ascending=False
).reset_index(drop=True)

display(resultats_cv_df.round(3))

In [ ]:
# Comparaison des F1-scores moyens en validation croisée
plt.figure(figsize=(10, 5))

sns.barplot(
    data=resultats_cv_df,
    x="Modèle",
    y="F1 moyen"
)

plt.title("Comparaison des modèles par validation croisée")
plt.xlabel("Modèle")
plt.ylabel("F1-score macro moyen")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. Sélection du modèle final

Le modèle présentant le meilleur F1-score macro moyen en validation croisée sera sélectionné pour une analyse plus approfondie. Ses performances seront ensuite examinées sur l'ensemble de test indépendant.

In [ ]:
# Sélection du modèle ayant le meilleur F1-score moyen
meilleur_nom = resultats_cv_df.iloc[0]["Modèle"]
meilleur_modele = modeles[meilleur_nom]

print("Modèle sélectionné :", meilleur_nom)

# Réentraînement sur l'ensemble d'entraînement
meilleur_modele.fit(X_train, y_train)

# Prédictions sur l'ensemble de test
y_pred = meilleur_modele.predict(X_test)

print(
    "Accuracy sur le test :",
    round(accuracy_score(y_test, y_pred), 3)
)

print(
    "F1-score macro sur le test :",
    round(f1_score(y_test, y_pred, average="macro"), 3)
)

## 9. Analyse des erreurs de classification

Le rapport de classification présente la précision, le rappel et le F1-score pour chaque espèce. La matrice de confusion permet d'identifier les espèces correctement classées et celles qui sont éventuellement confondues par le modèle.

In [ ]:
from sklearn.metrics import classification_report

# Rapport de classification
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

# Matrice de confusion
ordre_classes = sorted(y.unique())

matrice = confusion_matrix(
    y_test,
    y_pred,
    labels=ordre_classes
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    matrice,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=ordre_classes,
    yticklabels=ordre_classes
)

plt.title(f"Matrice de confusion — {meilleur_nom}")
plt.xlabel("Espèce prédite")
plt.ylabel("Espèce réelle")
plt.tight_layout()
plt.show()

## 10. Importance des caractéristiques

L'importance par permutation permet d'étudier l'influence des caractéristiques sur les prédictions du modèle retenu. Elle mesure la variation du score lorsque les valeurs d'une caractéristique sont mélangées dans les données de test.

In [ ]:
from sklearn.inspection import permutation_importance

# Calcul de l'importance par permutation
importance = permutation_importance(
    meilleur_modele,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=30,
    random_state=42
)

importance_df = pd.DataFrame({
    "Caractéristique": X.columns,
    "Importance moyenne": importance.importances_mean,
    "Écart-type": importance.importances_std
}).sort_values(
    by="Importance moyenne",
    ascending=False
).reset_index(drop=True)

display(importance_df.round(4))

In [ ]:
# Visualisation de l'importance des caractéristiques
plt.figure(figsize=(9, 5))

sns.barplot(
    data=importance_df,
    x="Importance moyenne",
    y="Caractéristique"
)

plt.title("Importance des caractéristiques par permutation")
plt.xlabel("Baisse moyenne du F1-score après permutation")
plt.ylabel("Caractéristique")
plt.tight_layout()
plt.show()

## 11. Prédiction sur une nouvelle fleur

Le modèle retenu est utilisé pour prédire l'espèce d'une nouvelle fleur à partir de ses quatre caractéristiques numériques.

In [ ]:
# Caractéristiques d'une nouvelle fleur
nouvelle_fleur = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=X.columns
)

# Prédiction de l'espèce
prediction = meilleur_modele.predict(nouvelle_fleur)

print("Caractéristiques de la nouvelle fleur :")
display(nouvelle_fleur)

print("Espèce prédite :", prediction[0])

## 13. Synthèse des résultats

Cette dernière étape rassemble les performances obtenues lors de l'évaluation et de la validation croisée afin de faciliter la comparaison finale des modèles.

In [ ]:
# Tableau récapitulatif des performances sur le jeu de test
synthese = resultats_df.merge(
    resultats_cv_df,
    on="Modèle"
)

display(synthese.round(3))

print("Modèle retenu :", meilleur_nom)

**Comparaison des modèles**

La régression logistique et l'arbre de décision ont obtenu une accuracy de 93,3 % et un F1-score macro de 0,933 sur les données de test. Random Forest a obtenu une accuracy de 90 % et un F1-score macro de 0,900.

En validation croisée, la régression logistique a obtenu le meilleur F1-score moyen, avec 0,958, contre 0,942 pour l'arbre de décision et 0,941 pour Random Forest.

**Choix du modèle final**

La régression logistique est retenue comme modèle final, car elle présente le meilleur F1-score moyen en validation croisée, tout en obtenant les mêmes performances que l'arbre de décision sur le jeu de test.

Ce résultat montre qu'un modèle relativement simple peut être performant sur le dataset Iris. Toutefois, les résultats ne démontrent pas que la régression logistique sera toujours supérieure aux autres algorithmes, notamment parce que le jeu de test est de petite taille.

**Limites et perspectives**

Le dataset Iris contient 150 observations et quatre caractéristiques numériques. Les performances mesurées doivent donc être interprétées avec prudence. Elles peuvent varier selon la séparation des données et les paramètres des modèles.

Pour améliorer cette étude, il serait intéressant d'optimiser les hyperparamètres, d'examiner les matrices de confusion des trois modèles et de tester les modèles sur de nouvelles données indépendantes.

**Conclusion**

Ce projet a permis d'appliquer les principales étapes d'un workflow de Machine Learning, de l'exploration des données à la comparaison et à l'interprétation des modèles. La régression logistique constitue le choix le plus pertinent parmi les trois modèles étudiés selon les résultats obtenus en validation croisée.